Public LB - 0.38961


In [1]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 106.0 MB/s eta 0:00:00
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 

In [2]:
import os
import pandas as pd
import shutil
from sklearn.model_selection import train_test_split

# ==========================================
# 1. Configuration Paths
# ==========================================
# Update this if your notebook/script is running in a different directory
source_dir = "/kaggle/input/competitions/road-vision/RoadVision_DUET/train" 
csv_path = os.path.join(source_dir, "train.csv")
images_dir = os.path.join(source_dir, "images")

output_dir = "/kaggle/working/"

# ==========================================
# 2. Create YOLO Directory Structure
# ==========================================
folders = [
    f"{output_dir}/images/train", f"{output_dir}/images/val",
    f"{output_dir}/labels/train", f"{output_dir}/labels/val"
]
for folder in folders:
    os.makedirs(folder, exist_ok=True)

# ==========================================
# 3. Process the CSV
# ==========================================
df = pd.read_csv(csv_path)

# The image_id column ends in .txt, so we extract the base filename
df['base_name'] = df['image_id'].apply(lambda x: os.path.splitext(x)[0])

# Get a list of unique images to split (80% train, 20% validation)
unique_images = df['base_name'].unique()
train_names, val_names = train_test_split(unique_images, test_size=0.2, random_state=42)

# Map base names to actual image files in the directory (handles .jpg, .png, etc.)
available_images = {
    os.path.splitext(f)[0]: f for f in os.listdir(images_dir) 
    if os.path.isfile(os.path.join(images_dir, f))
}

# ==========================================
# 4. Copy Files and Generate Labels
# ==========================================
def process_data(image_names, split_type):
    print(f"Processing {len(image_names)} images for {split_type}...")
    
    for base_name in image_names:
        if base_name not in available_images:
            continue  # Skip if the image file doesn't actually exist
            
        # Move Image
        img_filename = available_images[base_name]
        src_img = os.path.join(images_dir, img_filename)
        dst_img = os.path.join(output_dir, "images", split_type, img_filename)
        shutil.copy(src_img, dst_img)
        
        # Create corresponding .txt Label
        label_path = os.path.join(output_dir, "labels", split_type, f"{base_name}.txt")
        img_data = df[df['base_name'] == base_name]
        
        with open(label_path, 'w') as f:
            for _, row in img_data.iterrows():
                # YOLO format: class_id x_center y_center width height
                class_id = int(row['class_id'])
                x_c, y_c = row['x_center'], row['y_center']
                w, h = row['width'], row['height']
                
                f.write(f"{class_id} {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}\n")

process_data(train_names, 'train')
process_data(val_names, 'val')

# ==========================================
# 5. Generate data.yaml
# ==========================================
# Find max class ID to dynamically determine number of classes
num_classes = int(df['class_id'].max()) + 1
class_names = [f"class_{i}" for i in range(num_classes)]

yaml_content = f"""
path: {os.path.abspath(output_dir)}
train: images/train
val: images/val

nc: {num_classes}
names: {class_names}
"""

with open(os.path.join(output_dir, "data.yaml"), "w") as f:
    f.write(yaml_content.strip())

print("\nDataset preparation complete! You are ready to train.")

Processing 648 images for train...
Processing 162 images for val...

Dataset preparation complete! You are ready to train.


In [3]:
import os
import shutil
from ultralytics import YOLO

# 1. Load the larger model backbone for strong feature extraction
model = YOLO('yolov8l.pt') 



results = model.train(
    data='/kaggle/working/data.yaml',
    epochs=200,           # Much longer
    imgsz=960,           # Higher res — CCTV has small vehicles
    batch=8,
    patience=50,          # Early stopping
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=5,
    cos_lr=True,
    weight_decay=0.0005,
    dropout=0.1,
)

# 3. Post-Training Update: Safely copy the best model to the root workspace folder
try:
    # Extract the exact active directory where this run saved its artifacts
    active_save_dir = model.trainer.save_dir
    src_best_path = os.path.join(active_save_dir, 'weights', 'best.pt')
    
    # Destination path right at the root of your output files
    destination_path = '/kaggle/working/best_roadvision_model.pt'
    
    if os.path.exists(src_best_path):
        shutil.copy(src_best_path, destination_path)
        print(f"\n🎯 Export Complete! Your best weights are cleanly saved to: {destination_path}")
        print("You can download 'best_roadvision_model.pt' directly from your Kaggle Output pane.")
    else:
        print(f"\n⚠️ Training finished, but weights file was not found at: {src_best_path}")
except Exception as e:
    print(f"\n⚠️ Failed to copy model file: {e}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.1, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=